In [ ]:
import numpy as np

# 用 NumPy 从零实现两层神经网络 — 解决 XOR 问题

## 1. 任务背景

我们要学一个二分类函数 **XOR（异或）**：

| $x_1$ | $x_2$ | $y = x_1 \oplus x_2$ |
| :--:  | :--:  | :--: |
| 0     | 0     | 0 |
| 0     | 1     | 1 |
| 1     | 0     | 1 |
| 1     | 1     | 0 |

观察数据可以发现，**正类 (1) 和负类 (0) 在二维平面上是交叉分布的**，
无论怎么画一条直线都无法把 $\{ (0,1), (1,0) \}$ 与 $\{ (0,0), (1,1) \}$ 分开。
这说明 XOR **不是线性可分**的，单层感知机（无隐藏层）学不出来。

解决办法是引入一个**带激活函数的隐藏层**，让网络具有非线性能力。

## 2. 网络结构

$$
x \in \mathbb{R}^{2}
\;\xrightarrow{\,W_1,\,b_1\,}\;
z_1 \in \mathbb{R}^{4}
\;\xrightarrow{\sigma}\;
a_1 \in \mathbb{R}^{4}
\;\xrightarrow{\,W_2,\,b_2\,}\;
z_2 \in \mathbb{R}^{1}
\;\xrightarrow{\sigma}\;
\hat y \in (0,1)
$$

- 隐藏层 4 个神经元 + sigmoid 激活；
- 输出层 1 个神经元 + sigmoid，把 logits 压成概率。

## 3. 训练流程概览

1. **前向传播**：算 $z_1 \to a_1 \to z_2 \to \hat y$，并计算损失 $L$。
2. **反向传播**：用链式法则从输出往回算 $\partial L/\partial W_1, \partial L/\partial W_2$ 等梯度。
3. **参数更新**：$W \leftarrow W - \eta \cdot \partial L/\partial W$。
4. 重复 1~3 若干轮 (epoch)，直到损失不再下降。


In [ ]:
seed = 1
n_in = 2
n_hidden = 4
lr = 1.0

### 超参数

- `seed`：随机种子，保证结果可复现。
- `n_in=2`、`n_hidden=4`：输入维度和隐藏层神经元个数。
- `lr=1.0`（learning rate）：学习率，控制每一步参数更新的"步长"。  
  太小收敛慢，太大容易在最优解附近来回震荡甚至发散。


In [ ]:
# 训练数据：4 个样本 (N=4)，每个样本 2 个特征
# X.shape = (N, n_in) = (4, 2)
# y.shape = (N,)     = (4,)
X = np.array([[0,0],
              [0,1],
              [1,0],
              [1,1]])
y = np.array([0, 1, 1, 0])

## 初始化

### 形状约定

按"行=样本"的布局 (与 PyTorch / NumPy 矩阵乘法一致)：

$$
W^{(1)} \in \mathbb{R}^{n_{\text{in}} \times n_{\text{hidden}}}
\;=\; \mathbb{R}^{2 \times 4},
\qquad
W^{(2)} \in \mathbb{R}^{n_{\text{hidden}} \times 1}
\;=\; \mathbb{R}^{4 \times 1}
$$

- $W^{(1)}_{ij}$：从输入第 $i$ 维到隐藏层第 $j$ 个神经元的权重；
- $W^{(2)}_{ij}$：从隐藏层第 $i$ 个神经元到输出第 $j$ 个神经元的权重；
- 偏置 $b^{(1)} \in \mathbb{R}^{4}$，$b^{(2)} \in \mathbb{R}^{1}$。

### 初始化方式

使用均值为 0、标准差为 $1/\sqrt{n_{\text{in}}}$（前层神经元数）的高斯分布，
这其实就是 **Xavier 初始化**的简化版本，目的是让每一层激活值的方差大致保持在 1，
从而缓解 sigmoid 带来的梯度消失问题。

$$
W^{(1)}_{ij} \sim \mathcal N\!\left(0,\; \frac{1}{n_{\text{in}}}\right),
\quad
W^{(2)}_{ij} \sim \mathcal N\!\left(0,\; \frac{1}{n_{\text{hidden}}}\right)
$$

偏置全部初始化为 0。


In [ ]:
rng = np.random.default_rng(seed)

In [ ]:
W1 = rng.normal(0, 1/np.sqrt(n_in),size=(n_in, n_hidden))
W1.shape

In [ ]:
W1

In [ ]:
b1 = np.zeros(n_hidden)
b1

In [ ]:
W2 = rng.normal(0, 1/np.sqrt(n_hidden), size=(n_hidden, 1))
W2.shape

In [ ]:
W2

In [ ]:
b2 = np.zeros(1)
b2.shape

In [ ]:
b2

## 前向传播

### sigmoid 激活函数

$$
\sigma(z) \;=\; \frac{1}{1 + e^{-z}}
\quad\Rightarrow\quad
\sigma'(z) \;=\; \sigma(z)\bigl(1 - \sigma(z)\bigr)
$$

> 性质：$\sigma(z) \in (0,1)$ 单调递增；其导数用"激活后的值 $a=\sigma(z)$"  
> 表示为 $a(1-a)$，反向传播时直接用 `a` 算就行，不必再算 $z$。

### 隐藏层线性变换

$$
Z^{(1)} \;=\; X \, W^{(1)} + b^{(1)}
\;=\;
\underbrace{X}_{(N,2)}
\;\underbrace{W^{(1)}}_{(2,4)}
\;+\;
\underbrace{b^{(1)}}_{(4,)}
$$

形状对位：

| 量        | 形状        | 说明 |
| --- | --- | --- |
| $X$        | $(4, 2)$ | 4 个样本，每个 2 维 |
| $W^{(1)}$  | $(2, 4)$ | 输入到隐藏的权重 |
| $b^{(1)}$  | $(4,)$   | 隐藏层偏置（自动广播到 $(4,4)$ 后加到每一行） |
| $Z^{(1)}$  | $(4, 4)$ | 隐藏层"激活前"的输出 |

激活 $A^{(1)} = \sigma(Z^{(1)})$，形状不变 $(4,4)$。

### 输出层

$$
Z^{(2)} \;=\; A^{(1)} W^{(2)} + b^{(2)}
\;=\;
\underbrace{A^{(1)}}_{(4,4)}
\;\underbrace{W^{(2)}}_{(4,1)}
\;+\;
\underbrace{b^{(2)}}_{(1,)}
$$

$Z^{(2)}$ 形状 $(4,1)$，再过 sigmoid 得到 $\hat y = \sigma(Z^{(2)})$，  
最后 `.ravel()` 拉平成 $(4,)$ 与 $y$ 对齐。


In [ ]:
# 隐藏层线性变换：Z1 = X @ W1 + b1
# (4,2) @ (2,4) + (4,)  ->  (4,4)   (b1 沿行广播)
z1 = X @ W1 + b1
X.shape, W1.shape, b1.shape, z1.shape

In [ ]:
z1

In [ ]:
def sigmoid(z):
    # 为数值稳定用 np.clip
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def dsigmoid(a):
    """sigmoid 对 a 本身求导（a 已经是 sigmoid(z)）。

    由 σ'(z) = σ(z)·(1 − σ(z))，把 a = σ(z) 代入即可。
    直接用 a 算比用 z 算更省事，省一次 sigmoid 调用。
    """
    return a * (1.0 - a)

In [ ]:
# 隐藏层激活：A1 = σ(Z1)，形状仍然是 (4,4)
a1 = sigmoid(z1)
a1.shape

In [ ]:
a1

In [ ]:
# 输出层线性变换：Z2 = A1 @ W2 + b2
# (4,4) @ (4,1) + (1,)  ->  (4,1)
z2 = a1 @ W2 + b2
a1.shape, W2.shape, b2.shape, z2.shape

In [ ]:
z2

In [ ]:
# ŷ = σ(Z2)，.ravel() 把 (4,1) 拉平成 (4,)，便于和 y 做逐元素运算
y_pred = sigmoid(z2).ravel()
y_pred.shape

In [ ]:
y_pred

In [ ]:
# 二分类交叉熵损失（加 1e-8 防止 log(0)）
# L = -mean( y·log ŷ  +  (1-y)·log(1-ŷ) )
loss = -np.mean(y * np.log(y_pred + 1e-8) +
                (1 - y) * np.log(1 - y_pred + 1e-8))

### 损失函数：二分类交叉熵 (Binary Cross-Entropy)

对单个样本 $(x, y)$，$y \in \{0,1\}$，$\hat y \in (0,1)$：

$$
\ell\bigl(y, \hat y\bigr) \;=\;
-\,\Bigl[\,y \log \hat y \;+\; (1-y)\,\log(1-\hat y)\,\Bigr]
$$

对 $N$ 个样本取平均：

$$
L \;=\; \frac{1}{N} \sum_{i=1}^{N} \ell\bigl(y^{(i)}, \hat y^{(i)}\bigr)
\;=\;
-\frac{1}{N} \sum_{i=1}^{N}
\Bigl[\,y^{(i)} \log \hat y^{(i)} + (1-y^{(i)})\log(1-\hat y^{(i)})\,\Bigr]
$$

**为什么用交叉熵而不是 MSE？**  
当输出层用 sigmoid 时，$\partial L/\partial \hat y$ 在 $\hat y$ 接近 0 或 1 时  
被 $1/\hat y$ 或 $1/(1-\hat y)$ 项放大，**不会**像 MSE 那样因 sigmoid 导数小而梯度消失，  
训练更稳定。配合 sigmoid 输出层，交叉熵 + sigmoid 还有一个非常简洁的结论：

$$
\frac{\partial L}{\partial Z^{(2)}} \;=\; \hat y - y
$$

这正是后面反向传播第一行 `dz2 = y_pred - y` 的来源。


In [ ]:
loss

## 反向传播

### 整体思路

我们从损失 $L$ 出发，沿着前向传播的反方向，用**多元链式法则**把梯度从后往前推。

$$
L(\hat y, y)
\;\xleftarrow{\text{链式}}\;
Z^{(2)}
\;\xleftarrow{\text{链式}}\;
A^{(1)}
\;\xleftarrow{\text{链式}}\;
Z^{(1)}
\;\xleftarrow{\text{链式}}\;
W^{(1)}, b^{(1)}
$$

### 逐项推导

**(1) 输出层** —— 用"交叉熵 + sigmoid"那个极简结论：

$$
\frac{\partial L}{\partial Z^{(2)}} \;=\; \hat y - y \quad\Longrightarrow\quad
\boxed{dZ^{(2)} = \hat y - y}
$$

- $dZ^{(2)}$ 形状 $(N, 1)$，每行是一个样本的"误差信号"。

**(2) $W^{(2)}$, $b^{(2)}$ 的梯度**（$Z^{(2)} = A^{(1)} W^{(2)} + b^{(2)}$）：

$$
\frac{\partial L}{\partial W^{(2)}} \;=\; \frac{1}{N} (A^{(1)})^\top dZ^{(2)}
\;\in\; \mathbb R^{4 \times 1}
$$

$$
\frac{\partial L}{\partial b^{(2)}} \;=\; \frac{1}{N} \mathbf 1^\top dZ^{(2)}
\;\in\; \mathbb R^{1}
$$

**(3) 误差信号向隐藏层回传**：

$$
\frac{\partial L}{\partial A^{(1)}} \;=\; dZ^{(2)} \, (W^{(2)})^\top \quad \in\; \mathbb R^{N \times 4}
$$

**(4) 隐藏层激活导数**（$A^{(1)} = \sigma(Z^{(1)})$，$a(1-a)$ 是元素级）：

$$
dZ^{(1)} \;=\; dA^{(1)} \odot \sigma'(Z^{(1)}) \;=\; dA^{(1)} \odot A^{(1)} \odot \bigl(1 - A^{(1)}\bigr)
$$

**(5) $W^{(1)}$, $b^{(1)}$ 的梯度**（$Z^{(1)} = X W^{(1)} + b^{(1)}$）：

$$
\frac{\partial L}{\partial W^{(1)}} \;=\; \frac{1}{N} X^\top dZ^{(1)} \quad \in\; \mathbb R^{2 \times 4}
$$

$$
\frac{\partial L}{\partial b^{(1)}} \;=\; \frac{1}{N} \mathbf 1^\top dZ^{(1)} \quad \in\; \mathbb R^{4}
$$

### 为什么所有权重梯度公式长得都像 $X^\top dZ$？

在线性层 $Z = XW + b$ 里，$\partial L/\partial W$ 的第 $(i, j)$ 个元素  
$= \sum_n X_{n,i} \cdot dZ_{n,j}$，这正是 $(X^\top dZ)_{ij}$。  
所以"输入转置 × 输出梯度"是一个通用模式，在 PyTorch 里就是 `X.T @ dZ`。


In [ ]:
# 样本数，后面求平均梯度用
N = len(X)
N

In [ ]:
# 把 y_pred 从 (N,) 变回 (N,1)，便于和 W2 做矩阵乘法
y_pred = y_pred.reshape(-1, 1)
y_pred

In [ ]:
# dZ2 = ∂L/∂Z2 = ŷ - y       （交叉熵 + sigmoid 的经典结论）
# 形状 (N,1)
dz2 = (y_pred - y.reshape(-1, 1))
dz2

In [ ]:
# dW2 = (1/N) · A1.T @ dZ2
# (4,4).T @ (4,1) -> (4,1)，与 W2 同形
dW2 = (a1.T @ dz2) / N
dW2

In [ ]:
# db2 = dZ2 在样本维 (axis=0) 上的平均，形状 (1,)
db2 = dz2.mean(axis=0) 
db2

In [ ]:
# dA1 = dZ2 @ W2.T
# (N,1) @ (1,4) -> (N,4)，即 ∂L/∂A1
da1 = dz2 @ W2.T
da1

In [ ]:
# dZ1 = dA1 ⊙ σ'(A1)   (逐元素乘)
# dsigmoid(a) = a * (1 - a) 就是 σ'(Z1) 的"以 a 表达"形式
dz1 = da1 * dsigmoid(a1)
dz1

In [ ]:
# dW1 = (1/N) · X.T @ dZ1
# (2,4).T @ (4,4) -> (2,4)，与 W1 同形
dW1 = (X.T @ dz1) / N
dW1

In [ ]:
# db1 = dZ1 在样本维 (axis=0) 上的平均，形状 (4,)，与 b1 同形
db1 = dz1.mean(axis=0)
db1

## 梯度下降

有了梯度之后，沿着**损失下降最快的反方向**走一小步：

$$
W \;\leftarrow\; W - \eta\, \frac{\partial L}{\partial W},
\qquad
b \;\leftarrow\; b - \eta\, \frac{\partial L}{\partial b}
$$

其中 $\eta$ 就是学习率 `lr`。  
直观理解：

- $\partial L/\partial W$ 告诉我们在每个参数方向上"山坡有多陡"；
- 减去 $\eta$ 倍的坡度，就是"下山一步"；
- 反复迭代就能逐步逼近局部最优点（对凸问题就是全局最优点）。

> 提示：这里的"梯度"是**平均梯度**（已经除以 $N$），所以学习率的选择  
> 与 batch size 关系不大，便于不同数据量下复用同样的 `lr`。


In [ ]:
# 沿负梯度方向更新参数
W1 -= lr * dW1
b1 -= lr * db1
W2 -= lr * dW2
b2 -= lr * db2
W1, b1, W2, b2

In [ ]:
z1 = X @ W1 + b1
a1 = sigmoid(z1)
z2 = a1 @ W2 + b2
a2 = sigmoid(z2).ravel()
z1, a1, z2, a2

In [ ]:
loss = -np.mean(y * np.log(y_pred + 1e-8) +
    (1 - y) * np.log(1 - y_pred + 1e-8))
loss

In [ ]:
for ep in range(1, 1000 + 1):
    # ---- 前向 ----
    z1 = X @ W1 + b1          # (4,4)
    a1 = sigmoid(z1)          # (4,4)
    z2 = a1 @ W2 + b2         # (4,1)
    a2 = sigmoid(z2).ravel()  # (4,)
    y_pred =  a2
    
    # ---- 计算损失 ----
    loss = -np.mean(y * np.log(y_pred + 1e-8) +
                    (1 - y) * np.log(1 - y_pred + 1e-8))
    
    # 打印第 1 轮和每 100 轮的进度
    if ep % 100 == 0 or ep == 1:
        acc = (y_pred.round() == y).mean()
        print(f"epoch {ep:5d}  loss={loss:.4f}  acc={acc:.2f}")
                
    # ---- 反向 ----
    N = len(X)
    y_pred = y_pred.reshape(-1, 1)

    dz2 = (y_pred - y.reshape(-1, 1))   # 交叉熵+sigmoid 的极简结论
    dW2 = (a1.T @ dz2) / N
    db2 = dz2.mean(axis=0)

    da1 = dz2 @ W2.T
    dz1 = da1 * dsigmoid(a1)
    dW1 = (X.T @ dz1) / N
    db1 = dz1.mean(axis=0)

    # ---- 参数更新 (梯度下降) ----
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2    

## 训练循环

把前向 + 反向 + 更新打包成一个循环，跑若干个 **epoch**（一轮 = 把全部样本看一遍）。  
由于这里总共只有 4 个样本，相当于每次迭代都是 full-batch 梯度下降。